In [ ]:
import pandas as pd
import geopandas as gpd
import numpy as np
import json
import glob
import os
import itertools
from matplotlib import pyplot as plt
from collections import Counter
from concurrent.futures import ProcessPoolExecutor
from matplotlib import pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib.cm import tab10

In [ ]:
OUT_BASE = "/biostats_share/hillandr/data/WW_Mobility_2026_04_20/"
os.makedirs(os.path.dirname(OUT_BASE), exist_ok=True)

In [ ]:
# with open(CBG_MAP_FILE, "w") as f:
#     json.dump(geoid_to_sewershed, f)

In [ ]:
overlap = pd.read_csv("/biostats_share/hillandr/data/WW_Mobility_2025_10_09/meta/new_overlap_jan2025.csv", dtype={"GEOID": str}, index_col="GEOID")
overlap["pct_covered"] = overlap["PERCENTAGE"]/100
winner_sewershed = overlap.groupby("GEOID", group_keys=False).apply(lambda row: row.loc[(row.pct_covered == row.pct_covered.max()) & (row.pct_covered >= 0.002)])
geoid_to_sewershed = {k:v for k,v in zip(winner_sewershed.index, winner_sewershed.wwtp)}

In [ ]:
overlap.wwtp.nunique()

In [ ]:
with pd.option_context("display.max_rows", None):
    display(pd.DataFrame.from_records(((cbg, sewershed) for cbg, sewershed in geoid_to_sewershed.items()), columns=["CBG", "Sewershed"]).groupby("Sewershed").size())

In [ ]:
# Verify that we are using the correct CBG IDs (2018)
tiger_2018_shp = gpd.read_file("/biostats_share/hillandr/data/WW_Mobility_2025_10_09/tl_2018_08_bg/")
tiger_2018_geoids = set(tiger_2018_shp.GEOID.unique())

In [ ]:
def process_chunk_monthly(chunk: pd.DataFrame):
    data_chunk = chunk.copy()

    # Parse the device home areas field (JSON) into a dictionary.
    data_chunk["DEVICE_HOME_AREAS_D"] = data_chunk["DEVICE_HOME_AREAS"].map(json.loads)

    # Compute an intermediate column which is a tuple of two tuples containing the area keys and relative proportions of visits.
    data_chunk["DEVICE_HOME_AREAS_TMP_PARSE"] = data_chunk["DEVICE_HOME_AREAS_D"].map(lambda v: tuple(zip(*[(key, elem/sum(v.values())) for key, elem in v.items()])))
    # Extract the origin area IDs
    data_chunk["ORIGIN_AREA"] = data_chunk["DEVICE_HOME_AREAS_TMP_PARSE"].map(lambda e: e[0])
    # Extract the origin device fractions
    data_chunk["ORIGIN_DEVICE_FRAC"] = data_chunk["DEVICE_HOME_AREAS_TMP_PARSE"].map(lambda e: e[1])

    # Create one row for each element in the ORIGIN_AREA/ORIGIN_DEVICE_FRAC columns.
    data_chunk_explode = data_chunk[["DESTINATION_AREA", "DESTINATION_AREA_SEWERSHED", "DATE_RANGE_START", "ORIGIN_AREA", "DEVICE_COUNTS", "ORIGIN_DEVICE_FRAC"]].explode(column=["ORIGIN_AREA", "ORIGIN_DEVICE_FRAC"])
    # Map the extracted ORIGIN_AREA values 
    data_chunk_explode["ORIGIN_AREA_SEWERSHED"] = data_chunk_explode["ORIGIN_AREA"].map(geoid_to_sewershed)

    # Multiply the fraction by the total device count to get the estimated visitor count from the CBG.
    data_chunk_explode["ORIGIN_DEVICE_COUNT"] = data_chunk_explode["ORIGIN_DEVICE_FRAC"] * data_chunk_explode["DEVICE_COUNTS"]

    # Rows with the same destination and origin areas are 'home' devices, and should be counted.
    is_home_device = data_chunk_explode.DESTINATION_AREA == data_chunk_explode.ORIGIN_AREA
    # Rows with different destination and origin areas within the same sewershed are multiple counted, so we will exclude these.
    is_sewershed_visitor_device = data_chunk_explode.DESTINATION_AREA_SEWERSHED != data_chunk_explode.ORIGIN_AREA_SEWERSHED
    # Filtered data to only include these types of devices.
    data_chunk_filter = data_chunk_explode.loc[is_home_device | is_sewershed_visitor_device]

    # Aggregate up to DESTINATION_AREA level
    return data_chunk_filter.groupby(["DESTINATION_AREA_SEWERSHED", "DATE_RANGE_START"]).ORIGIN_DEVICE_COUNT.sum()

def process_chunk_daily(chunk: pd.DataFrame):
    data_chunk = chunk.copy()
    # Parse the list of stops by day.
    data_chunk["STOPS_BY_DAY_L"] = data_chunk["STOPS_BY_DAY"].map(json.loads)
    data_chunk["STOPS_BY_DAY_L_OFFSET"] = data_chunk["STOPS_BY_DAY_L"].map(lambda l: [pd.Timedelta(hours=24*i) for i in range(len(l))])
    data_chunk_explode = data_chunk.explode(column=["STOPS_BY_DAY_L", "STOPS_BY_DAY_L_OFFSET"])
    data_chunk_explode["DAY"] = data_chunk_explode["DATE_RANGE_START"] + data_chunk_explode["STOPS_BY_DAY_L_OFFSET"]
    return data_chunk_explode.groupby(["DESTINATION_AREA_SEWERSHED", "DAY"]).STOPS_BY_DAY_L.sum()

def process_chunk(path: str):
    # Read a data chunk.
    try:
        chunk = pd.read_parquet(path).rename(columns={"AREA": "DESTINATION_AREA"})
        # Convert DEVICE_COUNT to float
        chunk["DEVICE_COUNTS"] = chunk["DEVICE_COUNTS"].astype(float)
        # Map area IDs to sewershed IDs
        chunk["DESTINATION_AREA_SEWERSHED"] = chunk.DESTINATION_AREA.map(geoid_to_sewershed)
        # Verify that this data uses the 2018 CBG IDs
        assert len(set(chunk.DESTINATION_AREA.unique()) - tiger_2018_geoids) == 0
        # Drop all data which doesn't fall in a sewershed boundary
        chunk.dropna(subset=["DESTINATION_AREA_SEWERSHED"], inplace=True)
        # Process the chunk for monthly data
        monthly_processed_chunk = process_chunk_monthly(chunk)
        # Process the chunk for daily data
        daily_processed_chunk = process_chunk_daily(chunk)
        return monthly_processed_chunk, daily_processed_chunk
    except Exception as e:
        print(f"Error processing '{path}'.")
        raise e

In [ ]:
all_chunks = glob.glob(os.path.join(OUT_BASE, "neigh_patterns_plus/*.parquet"))
len(all_chunks)

In [ ]:
# Process in parallel
with ProcessPoolExecutor(max_workers=12) as pool:
    pool_out = list(pool.map(process_chunk, all_chunks))
# Retrieve monthly and daily data
monthly_out, daily_out = zip(*pool_out)

In [ ]:
monthly_df = pd.concat(monthly_out).sort_index().to_frame()

In [ ]:
os.makedirs(os.path.join(OUT_BASE, "processed"),exist_ok=True)

In [ ]:
monthly_df.to_csv(os.path.join(OUT_BASE, "processed/2026_04_20_monthly_devices.csv"))

In [ ]:
daily_df = pd.concat(daily_out).sort_index().to_frame()

In [ ]:
daily_df.to_csv(os.path.join(OUT_BASE, "processed/2026_04_20_daily_visits.csv"))

In [ ]:
daily_df.index.get_level_values("DAY").max()

In [ ]:
daily_df.index.get_level_values("DAY").month.value_counts()

In [ ]:
with pd.option_context("display.max_rows", None):
    display(monthly_df.index.get_level_values("DATE_RANGE_START").value_counts())

In [ ]:
monthly_df.index.get_level_values("DATE_RANGE_START").year.value_counts()

In [ ]:
old_monthly = pd.read_csv("/biostats_share/hillandr/data/WW_Mobility_2025_10_09/processed/2025_10_21_monthly_devices.csv", parse_dates=["DATE_RANGE_START"])
old_daily = pd.read_csv("/biostats_share/hillandr/data/WW_Mobility_2025_10_09/processed/2025_10_21_daily_visits.csv", parse_dates=["DAY"])
old_daily["STOPS_BY_DAY_L"] = old_daily["STOPS_BY_DAY_L"].astype(float)

In [ ]:
new_monthly = pd.read_csv("/biostats_share/hillandr/data/WW_Mobility_2026_03_03/processed/2026_03_03_monthly_devices.csv", parse_dates=["DATE_RANGE_START"])
new_monthly["DATE_RANGE_START"] = new_monthly["DATE_RANGE_START"].dt.tz_localize(None)

new_daily = pd.read_csv("/biostats_share/hillandr/data/WW_Mobility_2026_03_03/processed/2026_03_03_daily_visits.csv", parse_dates=["DAY"])
new_daily["DAY"] = new_daily["DAY"].dt.tz_localize(None)
new_daily["STOPS_BY_DAY_L"] = new_daily["STOPS_BY_DAY_L"].astype(float)

In [ ]:
monthly_merge = pd.merge(new_monthly, old_monthly, on=["DESTINATION_AREA_SEWERSHED", "DATE_RANGE_START"], how="outer", suffixes=("_new", "_old"))

In [ ]:
daily_merge = pd.merge(new_daily, old_daily, on=["DESTINATION_AREA_SEWERSHED", "DAY"], how="outer", suffixes=("_new", "_old"))

In [ ]:
with PdfPages("2026_03_03_data_comparison.pdf") as pdf:
    for wwtp in new_monthly["DESTINATION_AREA_SEWERSHED"].unique():
        monthly_tmp = monthly_merge.loc[monthly_merge.DESTINATION_AREA_SEWERSHED == wwtp]
        monthly_new = monthly_tmp.loc[monthly_tmp.index >= monthly_tmp.ORIGIN_DEVICE_COUNT_old.isna().idxmax()-1]
        daily_tmp = daily_merge.loc[daily_merge.DESTINATION_AREA_SEWERSHED == wwtp]
        daily_new = daily_tmp.loc[daily_tmp.index >= daily_tmp.STOPS_BY_DAY_L_old.isna().idxmax()-1]

        fig, ax = plt.subplots(figsize=(20,10), nrows=2, sharex=True)
        ax[0].set_title(f"Monthly Data Comparison for '{wwtp}'")
        ax[0].plot(monthly_tmp.DATE_RANGE_START, monthly_tmp.ORIGIN_DEVICE_COUNT_old, label="Original Data", color=tab10(7), alpha=0.2, linestyle="--")
        ax[0].plot(monthly_new.DATE_RANGE_START, monthly_new.ORIGIN_DEVICE_COUNT_new, label="New Dataset", color=tab10(1))
        ax[0].legend(fancybox=False, edgecolor="black")

        ax[1].set_title(f"Daily Data Comparison for '{wwtp}'")
        ax[1].plot(daily_tmp.DAY, daily_tmp.STOPS_BY_DAY_L_old, label="Old Dataset", color=tab10(7), alpha=0.2, linestyle="--")
        ax[1].plot(daily_new.DAY, daily_new.STOPS_BY_DAY_L_new, label="New Dataset", color=tab10(1))
        ax[1].legend(fancybox=False, edgecolor="black")
        
        pdf.savefig(fig)
        plt.close(fig)